# Pipeline de retrieve mejorado: extraccion + hibrido (denso+disperso) + rerank, BGE-m3 vs OpenAI

Este notebook junta, en un solo pipeline, las tres mejoras que la evaluacion con RAGAS fue justificando, y las aplica **igual** a `BAAI/bge-m3` y a `openai/text-embedding-3-large`, para comparar cuanto sube el techo de cada uno con el mismo tratamiento (no es una mejora "para BGE", es una mejora de retrieve en general que se prueba en los dos):

1. **Extraccion de tablas mejorada** (la que se dejo pendiente a proposito cuando probamos el hibrido solo, para aislar esa variable): tablas grandes se dividen, bloques tipo indice mal detectados como tabla se filtran. Motivo: 26 de las preguntas que fallan en BGE-m3 hibrido tambien fallan en OpenAI cs800 — la interseccion sugiere que una parte del problema es de extraccion, no de que embedding se use.
2. **Busqueda hibrida generalizada a los dos modelos**: BGE-m3 usa su propio vector disperso (lexical weights, via `FlagEmbedding`); OpenAI no tiene un vector disperso nativo, asi que se le da uno equivalente con **BM25** (via `rank_bm25`) sobre el mismo corpus. Ambos se fusionan con **Reciprocal Rank Fusion (RRF)**, implementado una sola vez en Python y reutilizado para los dos modelos (no el mecanismo nativo de Qdrant, para que el codigo de fusion sea identico en ambos casos y la comparacion sea justa).
3. **Reranking con cross-encoder** (`BAAI/bge-reranker-v2-m3`, multilingue) sobre un grupo de candidatos mas amplio (top ~20 denso + top ~20 disperso fusionados) antes de recortar a los `k=5` finales. Motivo: al subir k en el experimento de BGE hibrido, el recall mejoro bastante (0.736 -> 0.809) pero la precision cayo (0.701 -> 0.635) — es decir, ampliar la busqueda SI ayuda a encontrar mas, pero mete ruido. El rerank apunta a quedarse con lo mejor de ambos mundos: buscar ancho, pero devolver angosto y bien ordenado.

**Variables de entorno:** `OPENROUTER_API_KEY` (para el pipeline de OpenAI) y opcionalmente `HF_TOKEN` (no es necesario para BGE-m3, que no es gated).


In [ ]:
%pip install -q pymupdf4llm langchain-text-splitters langchain-core transformers FlagEmbedding rank_bm25 qdrant-client openai tiktoken

## 1. Extraccion unificada del PDF (con tablas divididas y filtradas)

Misma extraccion base de los notebooks anteriores, con las dos mejoras de tablas que habiamos probado y dejado pendientes de aplicar:
- `_es_tabla_de_contenido`: descarta bloques que la regex detecta como tabla pero son en realidad indices (lineas con "....." tipo lider de pagina).
- `_dividir_tabla_grande`: si una tabla real supera `LIMITE_CARACTERES_TABLA`, se divide en piezas mas chicas repitiendo encabezado + separador.


In [2]:
import re
import os
import uuid
from typing import Dict, List, Optional, Tuple

import pymupdf4llm

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_core.documents import Document

# Ruta del PDF final a procesar
pdf_path = "./archivo_fuente/7RAAC_Ingenierias Sistemas UV-24042025.pdf"

# Si una tabla extraida supera este tamano (en caracteres), se divide en piezas
# mas chicas en vez de quedar como un solo documento gigante.
LIMITE_CARACTERES_TABLA = 3000

c:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PATRON_TABLA_MD = re.compile(
    r'(^\|.*\|\s*\n'
    r'(?:\|[ \t]*:?-+:?[ \t]*\|.*\n)'
    r'(?:\|.*\|\s*\n?)+)',
    re.MULTILINE,
)

PATRON_LIDER_INDICE = re.compile(r'\.{4,}')


def _es_tabla_de_contenido(bloque_md: str) -> bool:
    lineas_con_lider = len(PATRON_LIDER_INDICE.findall(bloque_md))
    primera_linea = bloque_md.split("\n", 1)[0]
    columnas = primera_linea.count("|") - 1
    return lineas_con_lider >= 2 and columnas <= 1


def _dividir_tabla_grande(contenido_md: str, limite_caracteres: int = LIMITE_CARACTERES_TABLA) -> List[str]:
    if len(contenido_md) <= limite_caracteres:
        return [contenido_md]

    lineas = contenido_md.split("\n")
    tiene_encabezado = (
        len(lineas) >= 3
        and lineas[0].strip().startswith("|")
        and re.match(r'^\|[\s:|-]+\|?$', lineas[1].strip())
    )
    if not tiene_encabezado:
        return [contenido_md]

    encabezado, separador = lineas[0], lineas[1]
    prefijo = f"{encabezado}\n{separador}\n"
    filas = [l for l in lineas[2:] if l.strip()]

    partes: List[str] = []
    parte_actual: List[str] = []
    largo_actual = len(prefijo)
    for fila in filas:
        if largo_actual + len(fila) + 1 > limite_caracteres and parte_actual:
            partes.append(prefijo + "\n".join(parte_actual))
            parte_actual = []
            largo_actual = len(prefijo)
        parte_actual.append(fila)
        largo_actual += len(fila) + 1
    if parte_actual:
        partes.append(prefijo + "\n".join(parte_actual))

    return partes or [contenido_md]


def _separar_tablas_de_texto(texto: str, numero_pagina: Optional[int]) -> Tuple[str, List[Dict]]:
    tablas: List[Dict] = []
    contador = {"i": 0}

    def _reemplazo(match):
        bloque = match.group(0)
        if _es_tabla_de_contenido(bloque):
            return bloque
        contador["i"] += 1
        tablas.append({"numero_pagina": numero_pagina, "contenido_md": bloque.strip()})
        return f"\n[tabla {contador['i']} de la pagina {numero_pagina}]\n"

    texto_sin_tablas = PATRON_TABLA_MD.sub(_reemplazo, texto)
    return texto_sin_tablas, tablas


def extraer_documento_unificado(
    pdf_path: str,
    table_strategy: str = "lines_strict",
    header: bool = False,
    footer: bool = False,
) -> Dict:
    paginas = pymupdf4llm.to_markdown(
        pdf_path,
        page_chunks=True,
        table_strategy=table_strategy,
        header=header,
        footer=footer,
    )

    if isinstance(paginas, str):
        raise RuntimeError(
            "pymupdf4llm.to_markdown no devolvio una lista por pagina. "
            "Revisa tu version de pymupdf4llm / si tienes pymupdf-layout instalado."
        )

    partes_texto: List[str] = []
    mapa_paginas: List[Tuple[int, int, Optional[int]]] = []
    tablas_extraidas: List[Dict] = []
    offset_actual = 0

    for pagina in paginas:
        metadata_pagina = pagina.get("metadata", {})
        numero_pagina = metadata_pagina.get("page", metadata_pagina.get("page_number"))

        texto_pagina = pagina.get("text", "")
        texto_sin_tablas, tablas_pagina = _separar_tablas_de_texto(texto_pagina, numero_pagina)
        tablas_extraidas.extend(tablas_pagina)

        if partes_texto:
            partes_texto.append(" ")
            offset_actual += 1

        inicio = offset_actual
        partes_texto.append(texto_sin_tablas)
        offset_actual += len(texto_sin_tablas)
        fin = offset_actual

        mapa_paginas.append((inicio, fin, numero_pagina))

    for i, tabla in enumerate(tablas_extraidas):
        tabla["indice"] = i

    return {
        "texto_narrativo": "".join(partes_texto),
        "mapa_paginas": mapa_paginas,
        "tablas": tablas_extraidas,
    }


def _paginas_de_rango(mapa_paginas, inicio: int, fin: int) -> List[Optional[int]]:
    paginas = [
        num for (ini_p, fin_p, num) in mapa_paginas
        if ini_p < fin and fin_p > inicio and num is not None
    ]
    return paginas or [None]

In [4]:
extraido = extraer_documento_unificado(pdf_path)

documentos_tablas = []
tablas_divididas = 0
for t in extraido["tablas"]:
    partes = _dividir_tabla_grande(t["contenido_md"])
    if len(partes) > 1:
        tablas_divididas += 1
    for i, parte in enumerate(partes):
        documentos_tablas.append(Document(
            page_content=parte,
            metadata={
                "tipo": "tabla",
                "pagina": t["numero_pagina"],
                "fuente": pdf_path,
                "indice_tabla": t["indice"],
                "parte_tabla": i,
                "partes_totales_tabla": len(partes),
            },
        ))

print(f"Paginas procesadas: {len(extraido['mapa_paginas'])}")
print(f"Tablas detectadas: {len(extraido['tablas'])} ({tablas_divididas} divididas por tamano)")
print(f"Documentos de tabla resultantes: {len(documentos_tablas)}")

Paginas procesadas: 330
Tablas detectadas: 220 (3 divididas por tamano)
Documentos de tabla resultantes: 223


## 2. Chunkeo a chunk_size=800 (tokenizer propio de cada modelo)

Un solo tamano — 800 tokens, `chunk_overlap=0` — que fue el mejor en las 9 configuraciones originales para los tres modelos de embedding. Se genera una version de chunks para BGE-m3 (tokenizer XLM-R) y otra para OpenAI (tiktoken `cl100k_base`), porque los limites de token caen en lugares distintos del texto segun el tokenizer.


In [5]:
import tiktoken
from transformers import AutoTokenizer

CHUNK_SIZE = 800
CHUNK_OVERLAP = 0

TOKENIZER_BGE_M3 = AutoTokenizer.from_pretrained("BAAI/bge-m3")
_ENC_OPENAI = tiktoken.get_encoding("cl100k_base")


def contar_tokens_bge_m3(texto: str) -> int:
    return len(TOKENIZER_BGE_M3.encode(texto, add_special_tokens=False))


def contar_tokens_openai(texto: str) -> int:
    return len(_ENC_OPENAI.encode(texto))


def construir_chunks_texto(texto_narrativo, mapa_paginas, chunk_size, chunk_overlap, contador_tokens, pdf_path, nombre_modelo):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=contador_tokens,
        add_start_index=True,
    )
    chunks = splitter.create_documents([texto_narrativo])

    documentos = []
    for chunk in chunks:
        inicio = chunk.metadata.get("start_index", 0)
        fin = inicio + len(chunk.page_content)
        paginas = _paginas_de_rango(mapa_paginas, inicio, fin)
        paginas_validas = [p for p in paginas if p is not None]

        documentos.append(Document(
            page_content=chunk.page_content,
            metadata={
                "tipo": "texto",
                "pagina_inicio": min(paginas_validas) if paginas_validas else None,
                "pagina_fin": max(paginas_validas) if paginas_validas else None,
                "fuente": pdf_path,
                "chunk_size_config": chunk_size,
                "modelo_embedding": nombre_modelo,
            },
        ))
    return documentos


documentos_texto_bge = construir_chunks_texto(
    extraido["texto_narrativo"], extraido["mapa_paginas"], CHUNK_SIZE, CHUNK_OVERLAP,
    contar_tokens_bge_m3, pdf_path, "BAAI/bge-m3",
)
documentos_texto_openai = construir_chunks_texto(
    extraido["texto_narrativo"], extraido["mapa_paginas"], CHUNK_SIZE, CHUNK_OVERLAP,
    contar_tokens_openai, pdf_path, "openai/text-embedding-3-large",
)

print(f"Chunks de texto BGE-m3:  {len(documentos_texto_bge)}")
print(f"Chunks de texto OpenAI:  {len(documentos_texto_openai)}")

Chunks de texto BGE-m3:  204
Chunks de texto OpenAI:  249


## 3. Corpus con id estable + modelos denso/disperso por pipeline

Cada documento (texto o tabla) recibe un `id` (uuid) que se usa tanto como id del punto en Qdrant como llave del ranking disperso en memoria — asi los dos rankings (denso y disperso) se pueden fusionar directamente por id, sin tener que ir a buscar contenido a Qdrant en cada paso.


In [10]:
def construir_corpus(documentos_texto, documentos_tablas):
    corpus = documentos_texto + documentos_tablas
    ids = [str(uuid.uuid4()) for _ in corpus]
    return corpus, ids


corpus_bge, ids_bge = construir_corpus(documentos_texto_bge, documentos_tablas)
corpus_openai, ids_openai = construir_corpus(documentos_texto_openai, documentos_tablas)

mapa_id_a_documento_bge = dict(zip(ids_bge, corpus_bge))
mapa_id_a_documento_openai = dict(zip(ids_openai, corpus_openai))

print(f"Corpus BGE-m3:  {len(corpus_bge)} documentos")
print(f"Corpus OpenAI:  {len(corpus_openai)} documentos")

Corpus BGE-m3:  427 documentos
Corpus OpenAI:  472 documentos


In [6]:
from FlagEmbedding import BGEM3FlagModel

# use_fp16=True acelera en GPU; si corres solo en CPU y da problemas, cambia a False
modelo_bge = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)


def embeber_hibrido(textos, tam_lote=12, max_length=8192):
    """Devuelve (vectores_densos, vectores_dispersos) para una lista de textos.
    vectores_dispersos es una lista de dicts {token_id: peso}."""
    if not textos:
        return [], []
    salida = modelo_bge.encode(
        textos,
        batch_size=tam_lote,
        max_length=max_length,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    densos = [v.tolist() for v in salida["dense_vecs"]]
    dispersos = [
        {int(k): v for k, v in lw.items()}
        for lw in salida["lexical_weights"]
    ]
    return densos, dispersos

Loading weights: 100%|██████████| 391/391 [00:02<00:00, 135.92it/s]


In [7]:
import os
from openai import OpenAI

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("Define la variable de entorno OPENROUTER_API_KEY")

_cliente_openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
MODELO_OPENAI = "openai/text-embedding-3-large"


def embeber_openrouter(textos, nombre_modelo=MODELO_OPENAI, tam_lote=96):
    if not textos:
        return []
    vectores = []
    for i in range(0, len(textos), tam_lote):
        lote = textos[i:i + tam_lote]
        resp = _cliente_openrouter.embeddings.create(model=nombre_modelo, input=lote)
        vectores.extend([item.embedding for item in resp.data])
    return vectores

In [10]:
from rank_bm25 import BM25Okapi


def _tokenizar_bm25(texto: str) -> List[str]:
    return re.findall(r"\w+", texto.lower())

## 4. Indexar: vector denso en Qdrant, vector disperso en memoria

Qdrant solo guarda el vector denso de cada pipeline (una coleccion por modelo). El vector disperso (lexical weights de BGE, o el indice BM25 de OpenAI) se queda en memoria, alineado por posicion con `ids_bge` / `ids_openai` — el corpus es chico (unos cientos/miles de chunks de un solo PDF), asi que anotar el disperso contra todo el corpus en cada consulta es rapido, sin necesitar un indice aparte.


In [8]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

RUTA_QDRANT = "./qdrant_data_mejorado"  # carpeta propia, no choca con otros notebooks
client_qdrant = QdrantClient(path=RUTA_QDRANT)

COLECCION_BGE = "rag_mejorado_bge_m3_cs800"
COLECCION_OPENAI = "rag_mejorado_openai_cs800"


def asegurar_coleccion(client, nombre: str, dimension: int):
    if not client.collection_exists(nombre):
        client.create_collection(
            collection_name=nombre,
            vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
        )

In [15]:
# --- Pipeline BGE-m3: denso -> Qdrant, disperso -> memoria ---
textos_corpus_bge = [d.page_content for d in corpus_bge]
vectores_densos_bge, lexical_weights_bge = embeber_hibrido(textos_corpus_bge)

asegurar_coleccion(client_qdrant, COLECCION_BGE, dimension=1024)
puntos_bge = [
    PointStruct(id=doc_id, vector=vector, payload={"page_content": doc.page_content, **doc.metadata})
    for doc_id, doc, vector in zip(ids_bge, corpus_bge, vectores_densos_bge)
]
client_qdrant.upsert(collection_name=COLECCION_BGE, points=puntos_bge)
print(f"Coleccion \'{COLECCION_BGE}\': {len(puntos_bge)} puntos (denso). Disperso: {len(lexical_weights_bge)} en memoria.")

Inference Embeddings: 100%|██████████| 36/36 [06:08<00:00, 10.23s/it]


Coleccion 'rag_mejorado_bge_m3_cs800': 427 puntos (denso). Disperso: 427 en memoria.


In [16]:
# --- Pipeline OpenAI: denso -> Qdrant, disperso (BM25) -> memoria ---
textos_corpus_openai = [d.page_content for d in corpus_openai]
vectores_densos_openai = embeber_openrouter(textos_corpus_openai)

asegurar_coleccion(client_qdrant, COLECCION_OPENAI, dimension=3072)
puntos_openai = [
    PointStruct(id=doc_id, vector=vector, payload={"page_content": doc.page_content, **doc.metadata})
    for doc_id, doc, vector in zip(ids_openai, corpus_openai, vectores_densos_openai)
]
client_qdrant.upsert(collection_name=COLECCION_OPENAI, points=puntos_openai)

corpus_tokenizado_openai = [_tokenizar_bm25(t) for t in textos_corpus_openai]
bm25_openai = BM25Okapi(corpus_tokenizado_openai)

print(f"Coleccion \'{COLECCION_OPENAI}\': {len(puntos_openai)} puntos (denso). BM25 disperso construido sobre {len(corpus_tokenizado_openai)} documentos.")

Coleccion 'rag_mejorado_openai_cs800': 472 puntos (denso). BM25 disperso construido sobre 472 documentos.


## 5. Fusion RRF + reranking (funcion generica, reutilizada por los dos pipelines)

`_fusion_rrf` combina dos rankings de ids (denso y disperso) por posicion, igual que hicimos con `FusionQuery(fusion=Fusion.RRF)` de Qdrant en el notebook hibrido, pero implementado aqui en Python para poder aplicarlo exactamente igual a BGE (denso Qdrant + disperso lexical weights) y a OpenAI (denso Qdrant + disperso BM25) sin depender del vector disperso nativo de Qdrant.

Se trae un grupo mas amplio de candidatos (`TOP_N_DENSO` + `TOP_N_DISPERSO`, fusionados y recortados a `TAM_POOL_RERANK`) y el reranker decide el orden final de los `K_FINAL`.


In [15]:
from FlagEmbedding import FlagReranker

# use_fp16=True acelera en GPU; si corres solo en CPU y da problemas, cambia a False
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)

TOP_N_DENSO = 20      # candidatos de la rama densa antes de fusionar
TOP_N_DISPERSO = 20   # candidatos de la rama dispersa antes de fusionar
TAM_POOL_RERANK = 30  # tope de candidatos unicos que se le pasan al reranker
K_FINAL = 5
K_RRF = 60


def _fusion_rrf(rankings: List[List[str]], k_rrf: int = K_RRF) -> Dict[str, float]:
    """rankings: listas de ids ya ordenadas por relevancia (primera posicion = mas relevante)."""
    puntajes: Dict[str, float] = {}
    for ranking in rankings:
        for posicion, doc_id in enumerate(ranking):
            puntajes[doc_id] = puntajes.get(doc_id, 0.0) + 1.0 / (k_rrf + posicion + 1)
    return puntajes


def _producto_punto_disperso(pesos_a: Dict[int, float], pesos_b: Dict[int, float]) -> float:
    if len(pesos_a) > len(pesos_b):
        pesos_a, pesos_b = pesos_b, pesos_a
    return sum(peso * pesos_b.get(tok, 0.0) for tok, peso in pesos_a.items())


def ranking_disperso_bge(pesos_consulta: Dict[int, float], top_n: int = TOP_N_DISPERSO) -> List[str]:
    puntajes = [
        (doc_id, _producto_punto_disperso(pesos_consulta, lexical_weights_bge[i]))
        for i, doc_id in enumerate(ids_bge)
    ]
    puntajes.sort(key=lambda par: par[1], reverse=True)
    return [doc_id for doc_id, _ in puntajes[:top_n]]


def ranking_disperso_bm25(tokens_consulta: List[str], top_n: int = TOP_N_DISPERSO) -> List[str]:
    puntajes = bm25_openai.get_scores(tokens_consulta)
    indices_top = puntajes.argsort()[::-1][:top_n]
    return [ids_openai[i] for i in indices_top]


def buscar_hibrido_rerankeado(
    pregunta: str,
    coleccion_densa: str,
    vector_denso_pregunta,
    ranking_disperso_ids: List[str],
    mapa_id_a_documento: Dict[str, Document],
    k_final: int = K_FINAL,
):
    hits_densos = client_qdrant.query_points(
        collection_name=coleccion_densa,
        query=vector_denso_pregunta,
        limit=TOP_N_DENSO,
        with_payload=False,
    ).points
    ranking_denso_ids = [str(h.id) for h in hits_densos]

    puntajes_rrf = _fusion_rrf([ranking_denso_ids, ranking_disperso_ids])
    pool_ids = sorted(puntajes_rrf, key=puntajes_rrf.get, reverse=True)[:TAM_POOL_RERANK]

    if not pool_ids:
        return []

    pares = [[pregunta, mapa_id_a_documento[doc_id].page_content] for doc_id in pool_ids]
    scores_rerank = reranker.compute_score(pares, normalize=True)
    if isinstance(scores_rerank, float):
        scores_rerank = [scores_rerank]

    orden = sorted(zip(pool_ids, scores_rerank), key=lambda par: par[1], reverse=True)
    return orden[:k_final]

Loading weights: 100%|██████████| 393/393 [00:01<00:00, 310.99it/s]


## 6. Ejemplo rapido (sanity check antes del dataset completo)


In [18]:
pregunta_ejemplo = "¿cuál es el objetivo del programa?"

# --- BGE-m3 mejorado ---
vectores_densos_ej, dispersos_ej = embeber_hibrido([pregunta_ejemplo])
ranking_disperso_ej = ranking_disperso_bge(dispersos_ej[0])
resultado_bge = buscar_hibrido_rerankeado(
    pregunta_ejemplo, COLECCION_BGE, vectores_densos_ej[0], ranking_disperso_ej, mapa_id_a_documento_bge,
)
print("=== BGE-m3 mejorado ===")
for doc_id, score in resultado_bge:
    doc = mapa_id_a_documento_bge[doc_id]
    print(f"score_rerank={score:.4f} | tipo={doc.metadata.get('tipo')} | pagina={doc.metadata.get('pagina_inicio', doc.metadata.get('pagina'))}")
    print(doc.page_content[:300])
    print("---")

# --- OpenAI mejorado ---
vector_denso_ej_openai = embeber_openrouter([pregunta_ejemplo])[0]
ranking_disperso_ej_openai = ranking_disperso_bm25(_tokenizar_bm25(pregunta_ejemplo))
resultado_openai = buscar_hibrido_rerankeado(
    pregunta_ejemplo, COLECCION_OPENAI, vector_denso_ej_openai, ranking_disperso_ej_openai, mapa_id_a_documento_openai,
)
print("\n=== OpenAI mejorado ===")
for doc_id, score in resultado_openai:
    doc = mapa_id_a_documento_openai[doc_id]
    print(f"score_rerank={score:.4f} | tipo={doc.metadata.get('tipo')} | pagina={doc.metadata.get('pagina_inicio', doc.metadata.get('pagina'))}")
    print(doc.page_content[:300])
    print("---")

=== BGE-m3 mejorado ===
score_rerank=0.9682 | tipo=texto | pagina=39
El objetivo general del programa es formar profesionales en Ingeniería de Sistemas capaces de dar soluciones a problemas reales aplicando diversas áreas de la computación y destacándose por su formación científica, tecnológica, profesional y personal; con habilidad para mantenerse actualizados a med
---
score_rerank=0.9186 | tipo=texto | pagina=38
En el PEP se define la identidad del programa y se orientan las acciones formativas en el marco misional de la Universidad del Valle. El PEP Adopta los lineamientos curriculares del <u>Acuerdo 025 del 25 de septiembre de 2015</u> (ver Anexo 27) del Consejo Superior, que actualizó la Política Curricu
---
score_rerank=0.8782 | tipo=texto | pagina=300
[tabla 1 de la pagina 300]
El Consejo Superior de la Universidad del Valle estableció el Programa Institucional de Autoevaluación de la Calidad mediante la <u>Resolución 031 del 13 del 13 de junio de 2002</u> (ver Anexo 170) del C

## 7. Retrieval completo sobre el dataset de validacion (k=5, para ambos pipelines mejorados)

Cada pregunta se embebe **una sola vez por pipeline** (denso + disperso) y se reutiliza en la busqueda hibrida+rerank. Se guardan dos archivos, uno por pipeline, con el mismo esquema que los notebooks anteriores para que se puedan evaluar con el notebook de RAGAS.


In [13]:
import json
from pathlib import Path

RUTA_DATASET = "./validacion/dataset_validacion_final.json"
RUTA_SALIDA_BGE = "./validacion/resultados_retrieval_bge_m3_mejorado.json"
RUTA_SALIDA_OPENAI = "./validacion/resultados_retrieval_openai_mejorado.json"

with open(RUTA_DATASET, "r", encoding="utf-8") as f:
    dataset = json.load(f)

preguntas = dataset["pares"]
textos_preguntas = [p["pregunta"] for p in preguntas]
print(f"Preguntas cargadas: {len(preguntas)}")

Preguntas cargadas: 296


In [20]:
# Precomputar denso+disperso de las 296 preguntas, una sola vez por pipeline
vectores_densos_preg_bge, dispersos_preg_bge = embeber_hibrido(textos_preguntas)
vectores_densos_preg_openai = embeber_openrouter(textos_preguntas)
tokens_preg_bm25 = [_tokenizar_bm25(p) for p in textos_preguntas]

print("Consultas embebidas para ambos pipelines.")

Inference Embeddings: 100%|██████████| 25/25 [00:15<00:00,  1.63it/s]


Consultas embebidas para ambos pipelines.


In [21]:
def _construir_registro_resultado(pregunta_info, resultado, mapa_id_a_documento):
    return {
        "id": pregunta_info["id"],
        "pregunta": pregunta_info["pregunta"],
        "respuesta_esperada": pregunta_info.get("respuesta_esperada"),
        "evidencia_gold": pregunta_info.get("evidencia_gold"),
        "tipo": pregunta_info.get("tipo"),
        "dificultad": pregunta_info.get("dificultad"),
        "tipo_evidencia": pregunta_info.get("tipo_evidencia"),
        "seccion": pregunta_info.get("seccion"),
        "contextos_recuperados": [mapa_id_a_documento[doc_id].page_content for doc_id, _ in resultado],
        "metadatos_recuperados": [
            {
                "score_rerank": score,
                "tipo": mapa_id_a_documento[doc_id].metadata.get("tipo"),
                "pagina_inicio": mapa_id_a_documento[doc_id].metadata.get("pagina_inicio", mapa_id_a_documento[doc_id].metadata.get("pagina")),
                "pagina_fin": mapa_id_a_documento[doc_id].metadata.get("pagina_fin"),
            }
            for doc_id, score in resultado
        ],
    }


resultados_bge = []
for pregunta_info, v_denso, pesos_disp in zip(preguntas, vectores_densos_preg_bge, dispersos_preg_bge):
    ranking_disp = ranking_disperso_bge(pesos_disp)
    resultado = buscar_hibrido_rerankeado(
        pregunta_info["pregunta"], COLECCION_BGE, v_denso, ranking_disp, mapa_id_a_documento_bge,
    )
    resultados_bge.append(_construir_registro_resultado(pregunta_info, resultado, mapa_id_a_documento_bge))

salida_bge = {
    "modelo_embedding": "BAAI/bge-m3",
    "chunk_size": CHUNK_SIZE,
    "tipo_busqueda": "hibrida_dense_sparse_rrf_rerank",
    "coleccion": COLECCION_BGE,
    "k": K_FINAL,
    "experimento": "extraccion mejorada (tablas divididas + filtro TOC) + hibrido RRF + rerank cross-encoder",
    "dataset_fuente": dataset.get("documento"),
    "total_preguntas": len(preguntas),
    "resultados": resultados_bge,
}

Path(RUTA_SALIDA_BGE).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_SALIDA_BGE, "w", encoding="utf-8") as f: 
    json.dump(salida_bge, f, ensure_ascii=False, indent=2)

print(f"Listo. Guardado en {RUTA_SALIDA_BGE}")

Listo. Guardado en ./validacion/resultados_retrieval_bge_m3_mejorado.json


In [22]:
resultados_openai = []
for pregunta_info, v_denso, tokens_disp in zip(preguntas, vectores_densos_preg_openai, tokens_preg_bm25):
    ranking_disp = ranking_disperso_bm25(tokens_disp)
    resultado = buscar_hibrido_rerankeado(
        pregunta_info["pregunta"], COLECCION_OPENAI, v_denso, ranking_disp, mapa_id_a_documento_openai,
    )
    resultados_openai.append(_construir_registro_resultado(pregunta_info, resultado, mapa_id_a_documento_openai))

salida_openai = {
    "modelo_embedding": "openai/text-embedding-3-large",
    "chunk_size": CHUNK_SIZE,
    "tipo_busqueda": "hibrida_dense_bm25_rrf_rerank",
    "coleccion": COLECCION_OPENAI,
    "k": K_FINAL,
    "experimento": "extraccion mejorada (tablas divididas + filtro TOC) + hibrido RRF (denso + BM25) + rerank cross-encoder",
    "dataset_fuente": dataset.get("documento"),
    "total_preguntas": len(preguntas),
    "resultados": resultados_openai,
}

Path(RUTA_SALIDA_OPENAI).parent.mkdir(parents=True, exist_ok=True)
with open(RUTA_SALIDA_OPENAI, "w", encoding="utf-8") as f:
    json.dump(salida_openai, f, ensure_ascii=False, indent=2)

print(f"Listo. Guardado en {RUTA_SALIDA_OPENAI}")

Listo. Guardado en ./validacion/resultados_retrieval_openai_mejorado.json


In [23]:
# Sanity check rapido de los dos archivos
for ruta, resultados in [(RUTA_SALIDA_BGE, resultados_bge), (RUTA_SALIDA_OPENAI, resultados_openai)]:
    tamano_mb = Path(ruta).stat().st_size / (1024 * 1024)
    n_con_contexto = sum(1 for r in resultados if r["contextos_recuperados"])
    print(f"{ruta} ({tamano_mb:.2f} MB): {len(resultados)} preguntas, {n_con_contexto} con al menos 1 contexto recuperado")

./validacion/resultados_retrieval_bge_m3_mejorado.json (4.70 MB): 296 preguntas, 296 con al menos 1 contexto recuperado
./validacion/resultados_retrieval_openai_mejorado.json (4.01 MB): 296 preguntas, 296 con al menos 1 contexto recuperado


In [11]:
# --- Reconexion con las colecciones ya indexadas. NO escribe en Qdrant. ---
# Se arma el corpus en el mismo orden que construir_corpus (texto + tablas) y a cada documento
# se le asigna el id original del punto en Qdrant con el mismo payload exacto.
import json
from collections import defaultdict


def _clave_payload(payload):
    return json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str)


def reconectar_corpus(coleccion, documentos_texto, documentos_tablas):
    corpus = documentos_texto + documentos_tablas  # mismo orden que construir_corpus
    ids_por_clave = defaultdict(list)
    offset = None
    while True:
        puntos, offset = client_qdrant.scroll(
            collection_name=coleccion, limit=256, offset=offset, with_payload=True, with_vectors=False,
        )
        for p in puntos:
            ids_por_clave[_clave_payload(p.payload)].append(str(p.id))
        if offset is None:
            break

    n_puntos = sum(len(v) for v in ids_por_clave.values())
    if n_puntos != len(corpus):
        raise RuntimeError(f"'{coleccion}' tiene {n_puntos} puntos y el corpus {len(corpus)} (¿duplicados o extraccion distinta?)")

    ids = []
    for doc in corpus:
        candidatos = ids_por_clave.get(_clave_payload({"page_content": doc.page_content, **doc.metadata}))
        if not candidatos:
            raise RuntimeError(f"Hay un documento del corpus que no esta en '{coleccion}': la extraccion no reproduce lo indexado.")
        ids.append(candidatos.pop(0))
    return corpus, ids


corpus_bge, ids_bge = reconectar_corpus(COLECCION_BGE, documentos_texto_bge, documentos_tablas)
corpus_openai, ids_openai = reconectar_corpus(COLECCION_OPENAI, documentos_texto_openai, documentos_tablas)
mapa_id_a_documento_bge = dict(zip(ids_bge, corpus_bge))
mapa_id_a_documento_openai = dict(zip(ids_openai, corpus_openai))
print(f"Corpus reconectado: BGE {len(ids_bge)} | OpenAI {len(ids_openai)}")

# Parte dispersa en memoria, igual que en las celdas 15/16 pero sin upsert (es indexacion, no se mide)
_, lexical_weights_bge = embeber_hibrido([d.page_content for d in corpus_bge])  # ~6 min
bm25_openai = BM25Okapi([_tokenizar_bm25(d.page_content) for d in corpus_openai])
print("Disperso BGE y BM25 OpenAI listos.")

Corpus reconectado: BGE 427 | OpenAI 472


Inference Embeddings: 100%|██████████| 36/36 [05:49<00:00,  9.72s/it]

Disperso BGE y BM25 OpenAI listos.


In [16]:
# --- Medicion de latencia: BGE-m3 y OpenAI mejorados (hibrido + RRF + rerank) ---
import time
import numpy as np

N_CALENTAMIENTO = 5
RUTA_LAT_BGE = RUTA_SALIDA_BGE.replace(".json", "_latencia.json")
RUTA_LAT_OPENAI = RUTA_SALIDA_OPENAI.replace(".json", "_latencia.json")
RUTA_BASE_LATENCIA = "./validacion/resultados_retrieval_2_latencia.json"
ETAPAS = ["embedding_consulta", "busqueda_dispersa", "busqueda_densa_y_rrf", "rerank", "total_sin_rerank", "total"]


def _resumen(valores):
    a = np.asarray(valores, dtype=float)
    return {
        "media_ms": float(a.mean()),
        "mediana_ms": float(np.median(a)),
        "p95_ms": float(np.percentile(a, 95)),
        "min_ms": float(a.min()),
        "max_ms": float(a.max()),
        "desv_std_ms": float(a.std(ddof=1)),
    }


# Cronometro del rerank SIN modificar buscar_hibrido_rerankeado: se envuelve reranker.compute_score
if not hasattr(reranker, "_compute_score_sin_cronometro"):
    reranker._compute_score_sin_cronometro = reranker.compute_score
_ultimo_rerank_ms = [0.0]


def _compute_score_cronometrado(*args, **kwargs):
    t0 = time.perf_counter()
    salida = reranker._compute_score_sin_cronometro(*args, **kwargs)
    _ultimo_rerank_ms[0] = (time.perf_counter() - t0) * 1000
    return salida


reranker.compute_score = _compute_score_cronometrado


def _latencias(t_ini, t_emb, t_disp, t_fin):
    ms = lambda a, b: (b - a) * 1000
    rerank = _ultimo_rerank_ms[0]
    total = ms(t_ini, t_fin)
    return {
        "embedding_consulta": ms(t_ini, t_emb),
        "busqueda_dispersa": ms(t_emb, t_disp),
        "busqueda_densa_y_rrf": ms(t_disp, t_fin) - rerank,
        "rerank": rerank,
        "total_sin_rerank": total - rerank,
        "total": total,
    }


def consulta_bge(texto):
    _ultimo_rerank_ms[0] = 0.0
    t_ini = time.perf_counter()
    densos, dispersos = embeber_hibrido([texto])
    t_emb = time.perf_counter()
    ranking_disp = ranking_disperso_bge(dispersos[0])
    t_disp = time.perf_counter()
    resultado = buscar_hibrido_rerankeado(texto, COLECCION_BGE, densos[0], ranking_disp, mapa_id_a_documento_bge)
    t_fin = time.perf_counter()
    return resultado, _latencias(t_ini, t_emb, t_disp, t_fin)


def consulta_openai(texto):
    _ultimo_rerank_ms[0] = 0.0
    t_ini = time.perf_counter()
    vector = embeber_openrouter([texto])[0]
    t_emb = time.perf_counter()
    ranking_disp = ranking_disperso_bm25(_tokenizar_bm25(texto))
    t_disp = time.perf_counter()
    resultado = buscar_hibrido_rerankeado(texto, COLECCION_OPENAI, vector, ranking_disp, mapa_id_a_documento_openai)
    t_fin = time.perf_counter()
    return resultado, _latencias(t_ini, t_emb, t_disp, t_fin)


PIPELINES = {"bge": consulta_bge, "openai": consulta_openai}
MAPAS = {"bge": mapa_id_a_documento_bge, "openai": mapa_id_a_documento_openai}

# 1) Calentamiento (no se mide)
for texto in textos_preguntas[:N_CALENTAMIENTO]:
    for funcion in PIPELINES.values():
        funcion(texto)

# 2) Medicion: una consulta a la vez, pipelines intercalados con orden alternado
lat = {n: [] for n in PIPELINES}
contextos = {n: [] for n in PIPELINES}
nombres = list(PIPELINES)
for i, texto in enumerate(textos_preguntas):
    for n in (nombres if i % 2 == 0 else nombres[::-1]):
        resultado, l = PIPELINES[n](texto)
        lat[n].append(l)
        contextos[n].append([MAPAS[n][doc_id].page_content for doc_id, _ in resultado])
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(textos_preguntas)} preguntas")

reranker.compute_score = reranker._compute_score_sin_cronometro  # deja el reranker como estaba

# 3) Verificacion contra lo evaluado + guardado
config = {
    "bge": ("BAAI/bge-m3", COLECCION_BGE, "hibrida_dense_sparse_rrf_rerank", RUTA_SALIDA_BGE, RUTA_LAT_BGE),
    "openai": ("openai/text-embedding-3-large", COLECCION_OPENAI, "hibrida_dense_bm25_rrf_rerank", RUTA_SALIDA_OPENAI, RUTA_LAT_OPENAI),
}
resumenes = {}
for n, (modelo, coleccion, tipo_busqueda, ruta_eval, ruta_lat) in config.items():
    with open(ruta_eval, "r", encoding="utf-8") as f:
        previos = [r["contextos_recuperados"] for r in json.load(f)["resultados"]]
    iguales = sum(a == b for a, b in zip(contextos[n], previos))
    resumen = {e: _resumen([l[e] for l in lat[n]]) for e in ETAPAS}
    resumenes[n] = resumen

    print(f"\n=== {modelo} (contextos identicos a lo evaluado: {iguales}/{len(previos)}) ===")
    for e in ETAPAS:
        r = resumen[e]
        print(f"  {e:22s} media={r['media_ms']:8.1f} | mediana={r['mediana_ms']:8.1f} | p95={r['p95_ms']:8.1f} ms")

    salida_lat = {
        "modelo_embedding": modelo,
        "chunk_size": CHUNK_SIZE,
        "tipo_busqueda": tipo_busqueda,
        "coleccion": coleccion,
        "k": K_FINAL,
        "total_preguntas": len(preguntas),
        "metodo": f"una consulta a la vez; calentamiento de {N_CALENTAMIENTO} consultas; "
                  "pipelines intercalados con orden alternado; rerank medido envolviendo reranker.compute_score",
        "contextos_identicos_a_evaluacion": iguales,
        "resumen_latencias_ms": resumen,
        "latencias_por_pregunta_ms": [{"id": p["id"], **l} for p, l in zip(preguntas, lat[n])],
    }
    with open(ruta_lat, "w", encoding="utf-8") as f:
        json.dump(salida_lat, f, ensure_ascii=False, indent=2)
    print(f"  Guardado en {ruta_lat}")

# 4) Comparacion con la base densa cs800 (si ya corriste la medicion del notebook base)
if Path(RUTA_BASE_LATENCIA).exists():
    with open(RUTA_BASE_LATENCIA, "r", encoding="utf-8") as f:
        base = {(c["modelo_embedding"], c["chunk_size"]): c["resumen_latencias_ms"]["total"] for c in json.load(f)["configuraciones"]}
    print(f"\n{'pipeline':30s} | {'mediana':>8s} {'p95':>8s} | x base")
    for n, (modelo, *_resto) in config.items():
        b = base[(modelo, CHUNK_SIZE)]
        filas = [("base densa", b), ("hibrido sin rerank", resumenes[n]["total_sin_rerank"]), ("hibrido + rerank", resumenes[n]["total"])]
        for etiqueta, r in filas:
            print(f"{n + ' ' + etiqueta:30s} | {r['mediana_ms']:8.1f} {r['p95_ms']:8.1f} | {r['mediana_ms'] / b['mediana_ms']:5.2f}x")

  50/296 preguntas
  100/296 preguntas
  150/296 preguntas
  200/296 preguntas
  250/296 preguntas

=== BAAI/bge-m3 (contextos identicos a lo evaluado: 296/296) ===
  embedding_consulta     media=   288.7 | mediana=   286.6 | p95=   329.8 ms
  busqueda_dispersa      media=     2.6 | mediana=     2.5 | p95=     3.6 ms
  busqueda_densa_y_rrf   media=     2.7 | mediana=     2.7 | p95=     3.3 ms
  rerank                 media= 46084.6 | mediana= 46753.6 | p95= 48032.2 ms
  total_sin_rerank       media=   294.0 | mediana=   291.8 | p95=   335.8 ms
  total                  media= 46378.6 | mediana= 47049.8 | p95= 48316.1 ms
  Guardado en ./validacion/resultados_retrieval_bge_m3_mejorado_latencia.json

=== openai/text-embedding-3-large (contextos identicos a lo evaluado: 295/296) ===
  embedding_consulta     media=   999.7 | mediana=  1000.7 | p95=  1440.2 ms
  busqueda_dispersa      media=     2.2 | mediana=     2.2 | p95=     2.9 ms
  busqueda_densa_y_rrf   media=     7.6 | mediana=     7.

## Notas finales

- Los dos archivos (`resultados_retrieval_bge_m3_mejorado.json`, `resultados_retrieval_openai_mejorado.json`) quedan listos para el notebook de RAGAS. Agregalos a `RUTAS_RESULTADOS` en la celda de carga (junto a los que ya tenias) para comparar de una vez las 14 configuraciones: las 9 originales, BGE hibrido (k=5 y k=10), Gemma k=3, y estos dos pipelines mejorados.
- La comparacion que responde tu pregunta ("cuanto sube el techo") es: `BGE-m3 hibrido cs800` (0.701 / 0.736) vs `BGE-m3 mejorado`, y `OpenAI cs800` (0.742 / 0.831) vs `OpenAI mejorado`. Si el pipeline mejorado sube a los dos por un margen parecido, la conclusion es que las tres mejoras son value-add general de tu sistema RAG, no un parche especifico para BGE. Si sube mucho mas a BGE que a OpenAI, confirma que BGE se beneficia mas porque partia de mas atras (sobre todo en tablas).
- Costo/tiempo: el reranker corre sobre hasta `TAM_POOL_RERANK=30` pares por pregunta x 296 preguntas x 2 pipelines (~17,000 pares de rerank en total). Es un cross-encoder relativamente liviano, pero en CPU puede tardar varios minutos; si tienes GPU, `use_fp16=True` ya esta activado para aprovecharla.
- Si el pipeline mejorado no cierra tanto la brecha en tablas como se esperaba, el primer sospechoso es `TOP_N_DISPERSO` / `TOP_N_DENSO` (quizas el candidato correcto ni siquiera entra al pool antes del rerank) — subirlos es la palanca mas directa, a costa de mas latencia.
